In [ ]:
import torch
import stamina
from colpali_engine.models import ColModernVBert, ColModernVBertProcessor
from qdrant_client import QdrantClient
from qdrant_client.http import models
from pathlib import Path
import json

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

ModuleNotFoundError: No module named 'PIL.Image'

In [8]:
# 1. Load Model
print(f"Loading model {MODEL_NAME}...")
model = ColModernVBert.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32, # Matching reingest_colmodernvbert.py
    device_map=DEVICE,
    trust_remote_code=True
).eval()
processor = ColModernVBertProcessor.from_pretrained(MODEL_NAME)
print("Model loaded.")

# 2. Connect to Qdrant
client = QdrantClient(url=QDRANT_URL)
print(f"Connected to Qdrant at {QDRANT_URL}")

# Check collection status
collection_info = client.get_collection(COLLECTION_NAME)
print(f"Collection {COLLECTION_NAME} has {collection_info.points_count} points.")

Loading model ModernVBERT/colmodernvbert...
Model loaded.
Connected to Qdrant at http://localhost:6333
Collection colmodernvbert_collection has 3587 points.


In [9]:
# 3. Search Function
def search(query_text, top_k=2):
    """
    Search using ColModernVBert multi-vector search with score normalization.
    Menggunakan stamina.retry untuk menangani kegagalan koneksi sementara.
    """
    with torch.no_grad():
        # Process query
        query = processor.process_queries([query_text]).to(model.device)
        query_embedding = model(**query)
        
        # Ambil full embedding
        query_vec = query_embedding[0].cpu().float().numpy().tolist()
        num_query_tokens = len(query_vec)
    
    # Qdrant Query
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        using="initial",
        limit=top_k,
        search_params=models.SearchParams(
            hnsw_ef=128,
            quantization=models.QuantizationSearchParams(
                ignore=False,
                rescore=True,
            )
        )
    ).points
    
    # Normalisasi score
    return results

def print_results(results):
    for i, res in enumerate(results):
        norm_score = getattr(res, 'normalized_score', 0)
        print(f"Rank {i+1} | Score: {res.score:.2f} | Norm Score: {norm_score:.4f}")
        print(f"  PDF: {res.payload.get('pdf_name', 'N/A')}")
        print(f"  Page: {res.payload.get('page', 'N/A')}")
        if 'image_url' in res.payload:
            print(f"  URL: {res.payload['image_url']}")
        print("-" * 50)

In [10]:
query_text = "What are the common pests in apple trees?"
with torch.no_grad():
    batch_query = processor.process_queries([query_text]).to(
        model.device
    )
    query_embedding = model(**batch_query)
query_embedding

tensor([[[-0.0378, -0.1404, -0.0068,  ...,  0.0462, -0.0155,  0.0233],
         [-0.1264, -0.0585,  0.0604,  ..., -0.0212, -0.0346,  0.1244],
         [ 0.0584, -0.0693,  0.0635,  ..., -0.0693, -0.1016,  0.0639],
         ...,
         [-0.1481, -0.0817,  0.0229,  ...,  0.0443,  0.0714,  0.0346],
         [-0.1422, -0.0933,  0.0307,  ...,  0.0120,  0.0455,  0.0418],
         [ 0.2306, -0.0455,  0.0316,  ...,  0.2002, -0.0377, -0.0221]]],
       device='cuda:0')

In [11]:
query_embedding.shape

torch.Size([1, 22, 128])

In [13]:
# 4. Test Search
query = "Apple black rot disease management"
print(f"Searching for: '{query}'\n")

results = search(query)
print_results(results)

Searching for: 'Apple black rot disease management'

Rank 1 | Score: 11.85 | Norm Score: 0.0000
  PDF: _factsheet_plpath_fru_07
  Page: 1
  URL: https://thesis-assets.andyathsid.com/knowlege_base/_factsheet_plpath_fru_07/page_1.jpg
--------------------------------------------------
Rank 2 | Score: 11.42 | Norm Score: 0.0000
  PDF: Cedar-Apple_Rust
  Page: 4
  URL: https://thesis-assets.andyathsid.com/knowlege_base/Cedar-Apple_Rust/page_4.jpg
--------------------------------------------------
